# Graph Write and Read

Demonstrates the core document-graph read/write primitives against Neptune:

- `Node` / `Edge` models — typed graph elements
- `node_to_cypher` / `edge_to_cypher` — generate openCypher MERGE statements
- `batch_nodes_unwind` — optimized bulk write via UNWIND
- `DocumentGraphQueryEngine` — query tenant-scoped nodes

**Prerequisites:**
- `pip install graphrag-toolkit-document-graph[graphrag]`
- Neptune cluster endpoint (see 00-Setup)


In [ ]:
import os
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory
from graphrag_toolkit.document_graph import Node, Edge
from graphrag_toolkit.document_graph.graph_build import node_to_cypher, edge_to_cypher
from graphrag_toolkit.document_graph.query import DocumentGraphQueryEngine

GRAPH_STORE = os.environ.get("GRAPH_STORE", "neptune-db://<your-neptune-cluster-endpoint>:8182")
print("Imports OK")


## Write Nodes

Create typed nodes with tenant-scoped labels and write to Neptune.
Uses `batch_nodes_unwind` for efficient bulk writes via a single UNWIND query.


In [ ]:
graph_store = GraphStoreFactory.for_graph_store(GRAPH_STORE).__enter__()
TENANT = "notebook_demo"

# Create typed nodes
nodes = [
    Node(id="acc-1", labels=["Account"], properties={"name": "Production", "type": "PROD", "region": "us-east-1"}),
    Node(id="acc-2", labels=["Account"], properties={"name": "Development", "type": "DEV", "region": "us-west-2"}),
    Node(id="ps-1", labels=["PermissionSet"], properties={"name": "AdminAccess", "risk": "HIGH"}),
    Node(id="ps-2", labels=["PermissionSet"], properties={"name": "ReadOnly", "risk": "LOW"}),
]

for n in nodes:
    cypher, params = node_to_cypher(n, tenant_id=TENANT)
    graph_store.execute_query(cypher, params)

print(f"✅ Wrote {len(nodes)} nodes")


## Write Edges

Create typed relationships between existing nodes. Edge MATCH uses Neptune's
native `~id` index for O(1) vertex lookups.


In [ ]:
edges = [
    Edge(id="e1", source_id="ps-1", target_id="acc-1", label="ASSIGNED_TO"),
    Edge(id="e2", source_id="ps-2", target_id="acc-1", label="ASSIGNED_TO"),
    Edge(id="e3", source_id="ps-2", target_id="acc-2", label="ASSIGNED_TO"),
]

for e in edges:
    cypher, params = edge_to_cypher(e, tenant_id=TENANT)
    graph_store.execute_query(cypher, params)

print(f"✅ Wrote {len(edges)} edges")


## Query

Use `DocumentGraphQueryEngine` to read back tenant-scoped nodes.
The engine automatically applies the correct label format for multi-tenant isolation.


In [ ]:
engine = DocumentGraphQueryEngine(graph_store, tenant_id=TENANT)

print('All Accounts:')
for r in engine.get_nodes('Account'):
    print(f'  {r}')

print('\nHigh-risk Permission Sets:')
for r in engine.find_by_property('PermissionSet', 'risk', 'HIGH'):
    print(f'  {r}')

print('\nAssignments:')
for r in engine.get_relationships('PermissionSet', 'ASSIGNED_TO', 'Account'):
    print(f'  {r}')

# Clean up
gs.__exit__(None, None, None)
